In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
best_model = joblib.load(
    "../outputs/best_model.pkl"
)

print("Best model loaded successfully.")

Best model loaded successfully.


In [3]:
test_df = pd.read_csv(
    "../outputs/test_data.csv"
)

X_unseen = test_df.drop(
    "Revenue",
    axis=1
)

y_actual = test_df["Revenue"]

print("Unseen data loaded.")
print(X_unseen.shape)

Unseen data loaded.
(2441, 21)


In [4]:
predictions = best_model.predict(
    X_unseen
)

probabilities = best_model.predict_proba(
    X_unseen
)[:, 1]

In [5]:
prediction_results = X_unseen.copy()

prediction_results["Actual"] = y_actual.values

prediction_results["Predicted"] = predictions

prediction_results["Purchase Probability"] = (
    probabilities
)

prediction_results["Actual Label"] = (
    prediction_results["Actual"]
    .map({
        0: "No Purchase",
        1: "Purchase"
    })
)

prediction_results["Prediction Label"] = (
    prediction_results["Predicted"]
    .map({
        0: "No Purchase",
        1: "Purchase"
    })
)

prediction_results.head(10)

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,Weekend,TotalPages,TotalDuration,AveragePageDuration,EngagementScore,Actual,Predicted,Purchase Probability,Actual Label,Prediction Label
0,0,0.000000,0,0.0,12,482.500000,0.020000,0.040000,0.000000,0.0,...,False,12,482.500000,40.208333,482.500000,0,0,0.053882,No Purchase,No Purchase
1,2,287.200000,0,0.0,16,473.283333,0.000000,0.002222,0.000000,0.0,...,True,18,760.483333,42.249074,760.483333,0,0,0.031151,No Purchase,No Purchase
2,2,187.900000,4,100.1,29,2107.033333,0.000000,0.024242,0.000000,0.0,...,False,35,2395.033333,68.429524,2395.033333,0,0,0.022452,No Purchase,No Purchase
3,7,280.266667,0,0.0,11,329.683333,0.000000,0.023077,0.000000,0.0,...,False,18,609.950000,33.886111,609.950000,0,0,0.026666,No Purchase,No Purchase
4,0,0.000000,0,0.0,5,0.000000,0.200000,0.200000,0.000000,0.0,...,False,5,0.000000,0.000000,0.000000,0,0,0.009341,No Purchase,No Purchase
5,11,250.250000,0,0.0,169,8276.292565,0.002406,0.016052,8.174729,0.0,...,True,180,8526.542565,47.369681,8526.542565,0,1,0.702772,No Purchase,Purchase
6,0,0.000000,0,0.0,5,1377.250000,0.180000,0.186667,0.000000,0.0,...,False,5,1377.250000,275.450000,1377.250000,0,0,0.040082,No Purchase,No Purchase
7,10,519.000000,0,0.0,89,2577.960349,0.012852,0.037291,0.000000,0.0,...,False,99,3096.960349,31.282428,3096.960349,0,0,0.027722,No Purchase,No Purchase
8,0,0.000000,0,0.0,2,180.147059,0.033333,0.055556,0.000000,0.0,...,False,2,180.147059,90.073529,180.147059,0,0,0.047945,No Purchase,No Purchase
9,0,0.000000,0,0.0,6,161.666667,0.000000,0.050000,0.000000,0.0,...,False,6,161.666667,26.944444,161.666667,0,0,0.024089,No Purchase,No Purchase


In [6]:
final_prediction_view = prediction_results[
    [
        "Actual Label",
        "Prediction Label",
        "Purchase Probability"
    ]
].copy()

final_prediction_view.head(20)

,Actual Label,Prediction Label,Purchase Probability
0,No Purchase,No Purchase,0.053882
1,No Purchase,No Purchase,0.031151
2,No Purchase,No Purchase,0.022452
3,No Purchase,No Purchase,0.026666
4,No Purchase,No Purchase,0.009341
5,No Purchase,Purchase,0.702772
6,No Purchase,No Purchase,0.040082
7,No Purchase,No Purchase,0.027722
8,No Purchase,No Purchase,0.047945
9,No Purchase,No Purchase,0.024089


In [7]:
prediction_results.to_csv(
    "../outputs/final_predictions.csv",
    index=False
)

print("Final predictions saved successfully.")

Final predictions saved successfully.


In [8]:
new_customer = X_unseen.iloc[[0]].copy()

new_customer

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,TotalPages,TotalDuration,AveragePageDuration,EngagementScore
0,0,0.0,0,0.0,12,482.5,0.02,0.04,0.0,0.0,...,1,2,1,10,Returning_Visitor,False,12,482.5,40.208333,482.5


In [9]:
new_prediction = best_model.predict(
    new_customer
)[0]

new_probability = best_model.predict_proba(
    new_customer
)[0][1]

print("=" * 50)

print("NEW CUSTOMER PREDICTION")

print("=" * 50)

print(
    f"\nPurchase Probability: "
    f"{new_probability:.2%}"
)

if new_prediction == 1:
    
    print("Prediction: PURCHASE")
    
else:
    
    print("Prediction: NO PURCHASE")

NEW CUSTOMER PREDICTION

Purchase Probability: 5.39%
Prediction: NO PURCHASE


In [10]:
model_results = pd.read_csv(
    "../outputs/model_results.csv"
)

cv_results = pd.read_csv(
    "../outputs/cross_validation_results.csv"
)

best_model_result = model_results.iloc[0]

project_summary = pd.DataFrame({
    
    "Category": [
        "Problem Type",
        "Total Models Evaluated",
        "Cross Validation",
        "Hyperparameter Tuning",
        "Best Model",
        "Best Accuracy",
        "Best F1 Score",
        "Best ROC-AUC"
    ],
    
    "Result": [
        "Binary Classification",
        len(model_results),
        "5-Fold Stratified Cross Validation",
        "GridSearchCV and RandomizedSearchCV",
        best_model_result["Model"],
        round(best_model_result["Accuracy"], 4),
        round(best_model_result["F1 Score"], 4),
        round(best_model_result["ROC-AUC"], 4)
    ]
})

project_summary

,Category,Result
0,Problem Type,Binary Classification
1,Total Models Evaluated,6
2,Cross Validation,5-Fold Stratified Cross Validation
3,Hyperparameter Tuning,GridSearchCV and RandomizedSearchCV
4,Best Model,Gradient Boosting
5,Best Accuracy,0.9062
6,Best F1 Score,0.6859
7,Best ROC-AUC,0.9357


In [11]:
print("=" * 70)

print("PROJECT COMPLETED SUCCESSFULLY")

print("=" * 70)

print("\nProject Workflow Completed:")

steps = [
    
    "1. Dataset Understanding",
    
    "2. Exploratory Data Analysis",
    
    "3. Data Cleaning",
    
    "4. Feature Engineering",
    
    "5. Data Preprocessing",
    
    "6. Model Training",
    
    "7. Model Validation",
    
    "8. Cross Validation",
    
    "9. Hyperparameter Tuning",
    
    "10. Model Evaluation",
    
    "11. Best Model Selection",
    
    "12. Final Prediction"
]

for step in steps:
    
    print(step)

print("\nBest Model:")

print(best_model_result["Model"])

print(
    f"\nFinal ROC-AUC: "
    f"{best_model_result['ROC-AUC']:.4f}"
)

print("\nFinal Prediction file saved in:")

print("../outputs/final_predictions.csv")

PROJECT COMPLETED SUCCESSFULLY

Project Workflow Completed:
1. Dataset Understanding
2. Exploratory Data Analysis
3. Data Cleaning
4. Feature Engineering
5. Data Preprocessing
6. Model Training
7. Model Validation
8. Cross Validation
9. Hyperparameter Tuning
10. Model Evaluation
11. Best Model Selection
12. Final Prediction

Best Model:
Gradient Boosting

Final ROC-AUC: 0.9357

Final Prediction file saved in:
../outputs/final_predictions.csv
